# Review the paired L9 check on CPU

Use a CPU runtime. This notebook reads both L9 runs and their comparison from Drive. It does not load either model or run training.


In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/paired_L9_probe")
stem = "seq1_resnet50_seed0"
report_path = output / "paired_L9_comparison.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
assert report["status"] == "complete"
assert set(report["arms"]) == {"without_u3", "after_u3"}
saved = {}
for name in report["arms"]:
    folder = output / name
    saved[name] = {kind: json.loads((folder / f"{kind}_{stem}.json").read_text(encoding="utf-8"))
                   for kind in ("origin", "environment", "history", "costs", "summary")}
    checkpoint = folder / f"checkpoint_{stem}.pt"
    assert checkpoint.is_file() and checkpoint.stat().st_size > 0
    row = saved[name]["history"][-1]
    assert (row["action"], row["task"]) == ("learn", "9")
    assert len(saved[name]["history"]) == (3 if name == "without_u3" else 4)
    assert saved[name]["origin"]["starting_checkpoint_sha256"] == report["arms"][name]["source_sha256"]
    assert row["before"]["0"] == report["arms"][name]["task0_before"]
    assert row["after"]["0"] == report["arms"][name]["task0_after"]
    assert row["after"]["9"] == report["arms"][name]["task9_after"]
print("Saved runs checked:", ", ".join(saved))
print("Output:", output)


## Compare the change during L9

Task 0 starts at different accuracies in the two arms, so compare its loss during L9. Positive extra loss means task 0 fell more after U3. This one-seed check identifies the next issue to investigate; it is not a general result about the framework.


In [ ]:
import pandas as pd

rows = []
for name, arm in report["arms"].items():
    costs = saved[name]["costs"]["requests"]
    last_cost = next(row for row in costs if row["index"] == len(saved[name]["history"]) - 1)
    rows.append({"arm": name, "task0_before": arm["task0_before"],
                 "task0_after": arm["task0_after"],
                 "task0_loss_points": arm["task0_loss_points"],
                 "task9_after": arm["task9_after"],
                 "L9_seconds": last_cost["seconds"],
                 "git_commit": arm["git_commit"]})
display(pd.DataFrame(rows))
print("Extra task 0 loss after U3:", report["extra_task0_loss_after_u3"], "points")
print("Original full-run L9:", report["original_full_run_l9"])
print("After-U3 replay difference from full run, task 0 after L9:",
      report["arms"]["after_u3"]["task0_after"]
      - report["original_full_run_l9"]["task0_after"], "points")
